#### Подбор гиперпараметров LightGBM и выбор модели для инференса

In [160]:
import sys
import joblib
import numpy as np
import pandas as pd

from lightgbm import LGBMRegressor
from sklearn.model_selection import (
    GridSearchCV,
    TimeSeriesSplit
)

from pathlib import Path

sys.path.append(str(Path().resolve().parent))
from src.utils import get_project_root
from src.preprocessing import (
    preprocess_dataset,
    FEATURE_COLUMNS,
    TARGET
)
from src.splitting import split_dataset
from src.evaluation import evaluate_model

In [166]:
PROJECT_ROOT = get_project_root()
DATA_ROOT = PROJECT_ROOT / "data" / "raw"
ARTIFACTS_ROOT = PROJECT_ROOT / "artifacts" / "final_model"

In [132]:
raw_df = pd.read_csv(DATA_ROOT/"train.csv")

df = preprocess_dataset(raw_df)

In [133]:
X_train, X_test, y_train, y_test = split_dataset(df)

In [134]:
cv = TimeSeriesSplit(n_splits=3)

In [135]:
lgbm = LGBMRegressor(
    random_state=42,
    n_jobs=-1,
    verbose=-1
)

In [136]:
param_grid = {
    "n_estimators": [1000, 1100, 1200, 1300, 1500],
    "learning_rate": [0.02, 0.025, 0.03, 0.035, 0.04],
    "max_depth": [5, 6, 7],
    "num_leaves": [20, 25, 31, 35],
    "min_child_samples": [10, 15, 20, 25],
}

In [137]:
grid_search = GridSearchCV(
    estimator=lgbm,
    param_grid=param_grid,
    scoring="neg_root_mean_squared_error",
    cv=cv,
    n_jobs=-1,
    verbose=1,
)

grid_search.fit(X_train, y_train)

Fitting 3 folds for each of 1200 candidates, totalling 3600 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","LGBMRegressor...2, verbose=-1)"
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'learning_rate': [0.02, 0.025, ...], 'max_depth': [5, 6, ...], 'min_child_samples': [10, 15, ...], 'n_estimators': [1000, 1100, ...], ...}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'neg_root_mean_squared_error'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",TimeSeriesSpl...est_size=None)
,"verbose verbose: intControls the verbosity: the higher, t

In [142]:
print("Best grid search params:")
print(grid_search.best_params_)

print("Best CV RMSE:")
print(-grid_search.best_score_)

Best grid search params:
{'learning_rate': 0.04, 'max_depth': 5, 'min_child_samples': 10, 'n_estimators': 1300, 'num_leaves': 35}
Best CV RMSE:
76.72517493276943


In [143]:
grid_search_model = grid_search.best_estimator_

In [144]:
grid_search_model_metrics = evaluate_model(
    model=grid_search_model,
    X_train=X_train,
    X_test=X_test,
    y_train=y_train,
    y_test=y_test,
)

for metric, value in grid_search_model_metrics.items():
    print(f"{metric}: {value}")

train_MAE: 15.498020076182728
train_RMSE: 23.09228711642703
train_R2: 0.9807478429087167
train_MAPE: 0.3353029505114643
test_MAE: 37.65726351810106
test_RMSE: 56.58740322067076
test_R2: 0.9323708084750553
test_MAPE: 0.484121906793247


Модель, выбранная с помощью подбора гиперпараметров, показала более худшие результаты на тестовых данных по сравнению с лучшей моделью третьего эксперимента, имеет смысл детально сравнить их метрики качества.

In [145]:
previous_best_model = LGBMRegressor(
    n_estimators=1000,
    learning_rate=0.02,
    max_depth=7,
    num_leaves=31,
    min_child_samples=20,
    n_jobs=-1,
    verbose=-1,
)

previous_best_model_metrics = evaluate_model(
    model=previous_best_model,
    X_train=X_train,
    X_test=X_test,
    y_train=y_train,
    y_test=y_test
)

In [154]:
results_df = pd.DataFrame([grid_search_model_metrics, previous_best_model_metrics])
results_df.insert(loc=0, column="model", value=["grid_search_model", "previous_best_model"])
results_df = results_df.sort_values("test_RMSE", ignore_index=True)

results_df

,model,train_MAE,train_RMSE,train_R2,train_MAPE,test_MAE,test_RMSE,test_R2,test_MAPE
0,previous_best_model,17.271214,26.083985,0.975436,0.353441,36.165671,55.279079,0.935462,0.367997
1,grid_search_model,15.498020,23.092287,0.980748,0.335303,37.657264,56.587403,0.932371,0.484122


Итоги финального подбора гиперпараметров

Для модели LightGBM был выполнен детальный подбор гиперпараметров с использованием GridSearchCV и TimeSeriesSplit. Целью эксперимента являлось уточнение параметров в области значений, показавших наилучшие результаты в предыдущих экспериментах.

Несмотря на то, что найденная с помощью GridSearchCV конфигурация продемонстрировала более низкую ошибку на обучающей выборке, качество на тестовой выборке не улучшилось. Значение RMSE увеличилось с 55.28 до 56.59, а также наблюдается ухудшение других метрик качества.

Полученные результаты свидетельствуют о том, что дополнительное усложнение модели приводит к усилению переобучения и не улучшает её способность обобщать закономерности на новых данных.

Поэтому в качестве финальной модели была выбрана конфигурация LightGBM, полученная в третьем эксперименте с использованием RandomizedSearchCV, поскольку именно она показала наилучшее качество прогнозирования на тестовой выборке.

Так как данные, на которых обучалась и тестировалась модель имеют временной характер, разумно перед инференсом использовать весь объём информации.

Переобучим лучшую модель на всем датасете и сохраним ее.

In [167]:
X_full = df[FEATURE_COLUMNS]
y_full = df[TARGET]

In [168]:
final_model = previous_best_model

final_model.fit(X_full, y_full)

,boosting_type,'gbdt'
,num_leaves,31
,max_depth,7
,learning_rate,0.02
,n_estimators,1000
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [170]:
joblib.dump(final_model, ARTIFACTS_ROOT/"final_model.pkl")

['C:\\Users\\abdul\\gitrep\\a-ermakov\\project\\artifacts\\final_model\\final_model.pkl']

Покажем, что модель дает предсказания и готова к инференсу.

In [171]:
loaded_model = joblib.load(ARTIFACTS_ROOT/"final_model.pkl")

loaded_model.predict(X_full[:5])

array([32.43950673, 23.6404821 , 17.70087661,  2.5846562 , -3.48390605])